[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [APIs and JSON](https://johnfisher-ai.github.io/Python-Visual-Guides/apis-and-json.html)

# Sending Data &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The cell below rebuilds what the notebook set up: the practice API, with no plans yet. Run it first,
then the tasks in order, since tasks 3 and 5 use the plan task 1 creates.


In [1]:
import importlib
import sys
import urllib.request
import uuid
from pathlib import Path

import requests

PRACTICE_API = "https://raw.githubusercontent.com/johnfisher-ai/Python-Visual-Guides/main/notebooks/apis-and-json/practice_api.py"

if "google.colab" in sys.modules or not Path("practice_api.py").exists():
    urllib.request.urlretrieve(PRACTICE_API, "practice_api.py")    # in Colab, on every run

import practice_api
importlib.reload(practice_api)    # runs the file as it is now, not a copy imported earlier

BASE = practice_api.start()
print("ready:", BASE)


ready: http://127.0.0.1:8765


**1.** A new plan.


In [2]:
response = requests.post(f"{BASE}/network/plans", json={"name": "Harstad", "latitude": 68.80, "longitude": 16.54}, timeout=10)

print(response.status_code, response.headers["Location"], response.json())
harstad_url = f"{BASE}{response.headers['Location']}"


201 /network/plans/1 {'id': 1, 'name': 'Harstad', 'latitude': 68.8, 'longitude': 16.54}


It is the first plan since the practice API started, so its id is 1. The latitude comes back as
`68.8`: in JSON, as in Python, `68.80` and `68.8` are the same number.


**2.** Every problem with a plan.


In [3]:
response = requests.post(f"{BASE}/network/plans", json={"latitude": 68.0, "longitude": 200}, timeout=10)

print(response.status_code)
for problem in response.json()["problems"]:
    print(f"  {problem['field']}: {problem['problem']}")


422
  name: is required
  longitude: must be a number from -180 to 180


One problem for each field that was wrong, and no plan created, so the next plan still gets id 2.


**3.** A field added with PATCH.


In [4]:
print(requests.patch(harstad_url, json={"elevation_m": 20}, timeout=10).status_code)
print(requests.get(harstad_url, timeout=10).json())


200
{'id': 1, 'name': 'Harstad', 'latitude': 68.8, 'longitude': 16.54, 'elevation_m': 20}


The `PATCH` named one field, and the `GET` shows it added, with the other fields as they were.


**4.** A PUT to a plan that is not there.


In [5]:
whole = {"name": "Andenes", "latitude": 69.32, "longitude": 16.12}
response = requests.put(f"{BASE}/network/plans/999", json=whole, timeout=10)

print(response.status_code, response.json()["error"])


404 no plan has the id 999


This API's `PUT` replaces a plan that exists, and does not create one at an address the client
chooses. HTTP allows `PUT` to create, and some APIs do, which their documentation says. Here creating
is `POST`'s job, because the server chooses the id.


**5.** DELETE, twice.


In [6]:
first = requests.delete(harstad_url, timeout=10)
second = requests.delete(harstad_url, timeout=10)

print(first.status_code, second.status_code, "| GET:", requests.get(harstad_url, timeout=10).status_code)


204 404 | GET: 404


The first `DELETE` removed the plan, and the second found nothing to remove. The answers differ, and
the server is left the same way by either: without the plan.


**6.** One key, two requests, one plan.


In [7]:
karasjok = {"name": "Karasjok", "latitude": 69.47, "longitude": 25.51}
key = {"Idempotency-Key": str(uuid.uuid4())}
first = requests.post(f"{BASE}/network/plans", json=karasjok, headers=key, timeout=10)
second = requests.post(f"{BASE}/network/plans", json=karasjok, headers=key, timeout=10)

names = [plan["name"] for plan in requests.get(f"{BASE}/network/plans", timeout=10).json()]
print(first.status_code, second.status_code, "| replayed:", second.headers.get("Idempotent-Replayed"),
      "| plans named Karasjok:", names.count("Karasjok"))


201 201 | replayed: true | plans named Karasjok: 1


Both requests got `201`, and only the second was marked as replayed: the first created the plan, and
the second, with the same key and body, got the first's result back. `second.json()` is the same plan
as `first.json()`, id and all.


---

&#8592; **Back to:** [Sending Data](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/apis-and-json/13-sending-data.ipynb)  &nbsp;&middot;&nbsp;  [APIs and JSON Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/apis-and-json.html)
